In [ ]:
# ============================================================
# CELL 1 of 2 — SETUP. Paste this whole thing into one new cell and run it.
# Takes about a minute. Safe to run again any time Colab restarts.
# ============================================================

import os, sys, subprocess

# --- repo -----------------------------------------------------------------
if not os.path.isdir("/content/repo"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Wajeeha-Kamran/emr-assistant-backend.git",
                    "/content/repo"], check=True)
os.chdir("/content/repo")
sys.path.insert(0, os.getcwd())

# --- packages (skipped if already installed) -------------------------------
try:
    import whisper, pyannote.audio, soundfile, psutil  # noqa
    print("packages already present")
except ImportError:
    print("installing packages — if this line appears, RESTART THE SESSION")
    print("afterwards (Runtime > Restart session) and run this cell again.")
    subprocess.run("pip install -q openai-whisper 'pyannote.audio==4.0.7' "
                   "soundfile psutil && pip install -q -U numba",
                   shell=True, check=True)
    raise SystemExit("Installed. Now: Runtime > Restart session, then re-run this cell.")

# --- hardware --------------------------------------------------------------
import platform, json, torch, psutil

def hardware_report():
    info = {
        "platform": platform.platform(),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cpu_model": "unknown",
        "cpu_cores_physical": psutil.cpu_count(logical=False),
        "cpu_cores_logical": psutil.cpu_count(logical=True),
        "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 1),
        "gpu": None, "gpu_vram_gb": None, "cuda": torch.version.cuda,
    }
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                info["cpu_model"] = line.split(":", 1)[1].strip()
                break
    except Exception:
        pass
    if torch.cuda.is_available():
        info["gpu"] = torch.cuda.get_device_name(0)
        info["gpu_vram_gb"] = round(
            torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    return info

HARDWARE = hardware_report()
assert HARDWARE["gpu"], "No GPU. Runtime > Change runtime type > T4 GPU, then re-run."

# --- HF token --------------------------------------------------------------
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ["HF_TOKEN"], "HF_TOKEN secret is empty"

# --- the project's own metrics --------------------------------------------
from scripts.evaluate_accuracy import (
    parse_scripts, normalise, strip_numerics,
    word_error_rate, speaker_accuracy, audio_duration, SCRIPTS_MD, TARGET,
)
scripts = parse_scripts(SCRIPTS_MD)

# --- cost measurement ------------------------------------------------------
import time, gc, threading

class Measured:
    """Records VRAM, RAM, CPU and wall time for a block."""

    def __init__(self, label):
        self.label = label
        self.stats = {}

    def __enter__(self):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        self._proc = psutil.Process()
        self._cpu0 = sum(self._proc.cpu_times()[:2])
        self._rss_peak = self._proc.memory_info().rss
        self._stop = threading.Event()

        def sample():
            while not self._stop.wait(0.25):
                self._rss_peak = max(self._rss_peak, self._proc.memory_info().rss)

        self._sampler = threading.Thread(target=sample, daemon=True)
        self._sampler.start()
        self._t0 = time.time()
        return self

    def __exit__(self, *exc):
        self.stats["wall_s"] = round(time.time() - self._t0, 2)
        self._stop.set()
        self._sampler.join(timeout=1)
        self.stats["cpu_s"] = round(sum(self._proc.cpu_times()[:2]) - self._cpu0, 2)
        self.stats["ram_peak_gb"] = round(self._rss_peak / 1e9, 2)
        self.stats["vram_peak_gb"] = (
            round(torch.cuda.max_memory_allocated() / 1e9, 2)
            if torch.cuda.is_available() else None)
        return False

# --- harness ---------------------------------------------------------------
import pandas as pd

results = []
resource_rows = []

def benchmark(loader, engine, audio_dir, label, scripts=scripts):
    print(f"\n=== {label} — {os.path.basename(audio_dir)} ===")
    with Measured("load") as load:
        loader()
    print(f"  model load: {load.stats['wall_s']}s, VRAM {load.stats['vram_peak_gb']}GB")

    rows = []
    for n in sorted(scripts):
        wav = os.path.join(audio_dir, f"consult_{n}.wav")
        if not os.path.exists(wav):
            print(f"  script {n}: missing, skipped")
            continue

        ref_words, ref_spk = [], []
        for speaker, text in scripts[n]:
            w = normalise(text)
            ref_words.extend(w)
            ref_spk.extend([speaker] * len(w))

        with Measured(f"script{n}") as run:
            segments = engine(wav)

        hyp_words, hyp_spk = [], []
        for seg in segments:
            w = normalise(seg["text"])
            hyp_words.extend(w)
            hyp_spk.extend([seg["speaker_role"]] * len(w))

        wacc = max(0.0, 1 - word_error_rate(
            strip_numerics(ref_words), strip_numerics(hyp_words))) * 100
        correct, total = speaker_accuracy(ref_words, ref_spk, hyp_words, hyp_spk)
        spk = (correct / total * 100) if total else 0.0
        duration = audio_duration(wav)

        rows.append({
            "run": label, "audio_set": os.path.basename(audio_dir), "script": n,
            "word_acc": round(wacc, 1), "speaker_acc": round(spk, 1),
            "segments": len(segments), "ref_turns": len(scripts[n]),
            "audio_s": round(duration, 1), "infer_s": run.stats["wall_s"],
            "realtime_x": round(run.stats["wall_s"] / duration, 2) if duration else None,
            "cpu_s": run.stats["cpu_s"],
            "vram_peak_gb": run.stats["vram_peak_gb"],
            "ram_peak_gb": run.stats["ram_peak_gb"],
        })
        print(f"  script {n}: word {wacc:.1f}%  speaker {spk:.1f}%  "
              f"{run.stats['wall_s']}s  VRAM {run.stats['vram_peak_gb']}GB")

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    resource_rows.append({
        "run": label, "audio_set": os.path.basename(audio_dir),
        "load_s": load.stats["wall_s"], "load_vram_gb": load.stats["vram_peak_gb"],
        "vram_peak_gb": df.vram_peak_gb.max(), "ram_peak_gb": df.ram_peak_gb.max(),
        "cpu_s_mean": round(df.cpu_s.mean(), 1),
        "infer_s_mean": round(df.infer_s.mean(), 1),
        "realtime_x_mean": round(df.realtime_x.mean(), 2),
        "word_acc_mean": round(df.word_acc.mean(), 1),
        "speaker_acc_mean": round(df.speaker_acc.mean(), 1),
    })
    print(f"  MEAN  word {df.word_acc.mean():.1f}%  speaker {df.speaker_acc.mean():.1f}%"
          f"  | peak VRAM {df.vram_peak_gb.max()}GB  RT x{df.realtime_x.mean():.2f}")
    results.append(df)
    return df

# --- engines ---------------------------------------------------------------
import whisper
from pyannote.audio import Pipeline

_models = {}

def load_whisper(name):
    if name not in _models:
        _models[name] = whisper.load_model(name, device="cuda")
    return _models[name]

def load_pyannote():
    if "pyannote" not in _models:
        p = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1",
                                     token=os.environ["HF_TOKEN"])
        p.to(torch.device("cuda"))
        _models["pyannote"] = p
    return _models["pyannote"]

def assign_roles(segments):
    """Label the more question-asking cluster DOCTOR. Mirrors DiarizationService."""
    counts = {}
    for s in segments:
        counts[s["speaker"]] = counts.get(s["speaker"], 0) + s["text"].count("?")
    if not counts:
        return segments
    doctor = max(counts, key=counts.get)
    for s in segments:
        s["speaker_role"] = "DOCTOR" if s["speaker"] == doctor else "PATIENT"
    return segments

def words_to_turns(words, spans):
    """Give each word the speaker whose turn covers it, then merge runs."""
    def speaker_at(t):
        for start, end, spk in spans:
            if start <= t <= end:
                return spk
        return min(spans, key=lambda s: min(abs(s[0] - t), abs(s[1] - t)))[2] if spans else "A"

    merged, current = [], None
    for w in words:
        spk = speaker_at((w["start"] + w["end"]) / 2)
        if current is None or current["speaker"] != spk:
            if current:
                merged.append(current)
            current = {"speaker": spk, "text": w["word"]}
        else:
            current["text"] += w["word"]
    if current:
        merged.append(current)
    return assign_roles(merged)

def _to_spans(turns):
    """pyannote 4.x returns DiarizeOutput; 3.x returned the Annotation itself."""
    if hasattr(turns, "itertracks"):
        src = turns
    else:
        src = None
        for attr in ("speaker_diarization", "diarization", "annotation",
                     "exclusive_speaker_diarization"):
            v = getattr(turns, attr, None)
            if v is not None and hasattr(v, "itertracks"):
                src = v
                break
        if src is None:
            raise RuntimeError(
                f"No Annotation on {type(turns).__name__}: "
                f"{[a for a in dir(turns) if not a.startswith('_')]}")
    return [(t.start, t.end, spk) for t, _, spk in src.itertracks(yield_label=True)]

def whisper_pyannote(wav, model_name):
    asr = load_whisper(model_name).transcribe(wav, word_timestamps=True)
    words = [w for seg in asr.get("segments", []) for w in seg.get("words", [])]
    return words_to_turns(words, _to_spans(load_pyannote()(wav)))

# --- checkpointing (so a Colab restart never costs you a run) ---------------
CKPT_DETAIL = "/content/ckpt_detail.csv"
CKPT_RESOURCE = "/content/ckpt_resource.csv"

def save_checkpoint():
    if results:
        pd.concat([x for x in results if not x.empty],
                  ignore_index=True).to_csv(CKPT_DETAIL, index=False)
    if resource_rows:
        pd.DataFrame(resource_rows).to_csv(CKPT_RESOURCE, index=False)
    print(f"checkpoint saved ({len(resource_rows)} runs)")

def load_checkpoint():
    """Put previously saved runs back after a restart."""
    if os.path.exists(CKPT_DETAIL):
        prev = pd.read_csv(CKPT_DETAIL)
        results.clear(); results.append(prev)
        print(f"restored: {sorted(prev.run.unique())}")
    if os.path.exists(CKPT_RESOURCE):
        resource_rows.clear()
        resource_rows.extend(pd.read_csv(CKPT_RESOURCE).to_dict("records"))

load_checkpoint()

print()
print("SETUP OK")
print(json.dumps(HARDWARE, indent=2))
print(f"{len(scripts)} reference scripts: {sorted(scripts)}")
print("Now run CELL 2.")

packages already present

SETUP OK
{
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "python": "3.12.13",
  "torch": "2.11.0+cu128",
  "cpu_model": "Intel(R) Xeon(R) CPU @ 2.00GHz",
  "cpu_cores_physical": 1,
  "cpu_cores_logical": 2,
  "ram_total_gb": 13.6,
  "gpu": "Tesla T4",
  "gpu_vram_gb": 15.6,
  "cuda": "12.8"
}
4 reference scripts: [1, 2, 3, 4]
Now run CELL 2.


In [ ]:
# ============================================================
# CELL 11 — NOTEBOOK A. Dump Whisper base.en's words.
#
# Run PASTE_1_setup.py first (that cell defines everything this one uses).
# Takes about 2 minutes. Produces /content/handoff_base.json — download it.
#
# This is the same dump as before but with base.en instead of medium, because
# the combination being tested is base.en + Sortformer. pyannote is not loaded
# at all this time; the diarizer under test is Sortformer.
# ============================================================

import json

AUDIO_DIR = "docs/evidence/human_distinct"

load_whisper("base.en")

dump = {"hardware": HARDWARE, "audio_dir": AUDIO_DIR, "asr": "base.en", "scripts": {}}

for n in sorted(scripts):
    wav = f"{AUDIO_DIR}/consult_{n}.wav"

    with Measured(f"whisper{n}") as mw:
        asr = load_whisper("base.en").transcribe(wav, word_timestamps=True)

    words = [{"start": float(w["start"]), "end": float(w["end"]), "word": w["word"]}
             for seg in asr.get("segments", []) for w in seg.get("words", [])]

    dump["scripts"][str(n)] = {
        "audio_s": audio_duration(wav),
        "whisper_base_words": words,
        "whisper_base_stats": mw.stats,
    }
    print(f"script {n}: {len(words)} words in {mw.stats['wall_s']}s "
          f"(VRAM {mw.stats['vram_peak_gb']}GB)")

with open("/content/handoff_base.json", "w") as f:
    json.dump(dump, f)

import os
print(f"\nWritten /content/handoff_base.json "
      f"({os.path.getsize('/content/handoff_base.json')/1e6:.1f} MB)")
print("Download it, then move to Notebook B.")

100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 134MiB/s]


script 1: 217 words in 9.67s (VRAM 0.49GB)
script 2: 214 words in 3.14s (VRAM 0.49GB)
script 3: 177 words in 3.22s (VRAM 0.49GB)
script 4: 339 words in 4.94s (VRAM 0.49GB)

Written /content/handoff_base.json (0.0 MB)
Download it, then move to Notebook B.
